# Ablasyon — her değişiklik tek başına (Colab)

Çalışma zamanı: **T4 GPU**. Kurallar `docs/protokol_ablasyon.md` dosyasındadır.

Bir oturumda **tek model** çalıştırılır; aşağıdaki `MODEL` değişkenini seç. Takımda herkes farklı bir modeli kendi Colab hesabında paralel çalıştırabilir.

Her model için 4 koşu (hepsi `--amp`):

| Koşu | Değişen |
|---|---|
| temel | hiçbiri: 224 piksel, hafif artırma, normal örnekleme |
| 384 | yalnız çözünürlük 384 |
| guclu | yalnız güçlü artırma |
| denge | yalnız sınıf dengeli örnekleme |

TTA her koşuda ayrıca ölçülür. "Hepsi birlikte" koşusu Aşama 2'den hazırdır.

Her koşunun sonunda zip iner. Oturum koparsa kurulum hücrelerini çalıştır, indirdiğin zip'leri "Kaldığın yerden devam" hücresiyle yükle; biten koşular atlanır.

In [ ]:
MODEL = 'convnext_tiny'  # resnet50, efficientnet_b0, convnext_tiny, densenet121
assert MODEL in ('resnet50', 'efficientnet_b0', 'convnext_tiny', 'densenet121'), MODEL

In [ ]:
import torch
print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
assert torch.cuda.is_available(), 'GPU kapalı. Çalışma zamanını T4 yapıp yeniden başlat.'

In [ ]:
%cd /content
!rm -rf Final-project
!git clone https://github.com/bernademirer/Final-project.git
%cd /content/Final-project
!git log --oneline -3
from pathlib import Path
assert 'guclu' in Path('src/train.py').read_text(encoding='utf-8'), "GitHub'daki kod eski."

In [ ]:
!pip -q install -r requirements.txt datasets

In [ ]:
import pandas as pd
from datasets import load_dataset

splits = pd.read_csv('data/splits.csv')
assert len(splits) == 1457
assert (splits.kume == 'test').sum() == 292
assert splits.file_name.nunique() == 1457

out = Path('data/images')
out.mkdir(parents=True, exist_ok=True)
need = set(splits.file_name) - {p.name for p in out.glob('*.jpg')}
if need:
    for row in load_dataset('Layered-Labs/ACNE04')['train']:
        if row['file_name'] in need:
            row['image'].save(out / row['file_name'])
print('jpg', len(list(out.glob('*.jpg'))))
assert len(list(out.glob('*.jpg'))) == 1457

### Kaldığın yerden devam (yalnız oturum koptuysa)

Bu modelin daha önce indirdiğin `abl_<model>_<koşu>.zip` dosyalarını (birden fazla seçebilirsin) yükle.

In [ ]:
from google.colab import files
import zipfile
up = files.upload()
for name in up:
    zipfile.ZipFile(name).extractall('/content/Final-project')
!ls results | grep _abl_

## Yardımcı fonksiyon

In [ ]:
import json
import os
import re
import subprocess
import zipfile
from google.colab import files

VARYANTLAR = {
    'temel': '--aug on',
    '384': '--aug on --image-size 384',
    'guclu': '--aug guclu',
    'denge': '--aug on --balanced',
}


def calistir(cmd):
    print('$', cmd, flush=True)
    env = {**os.environ, 'PYTHONUNBUFFERED': '1'}
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    lines = []
    for line in p.stdout:
        print(line, end='', flush=True)
        lines.append(line)
    if p.wait():
        raise RuntimeError('komut hata verdi: ' + cmd)
    return ''.join(lines)


def olcum(cikti):
    d = {k: float(v) for k, v in re.findall(r'^(dogruluk|macro_f1|recall_\d) ([0-9.]+)', cikti, re.M)}
    return {'dogruluk': d['dogruluk'], 'macro_f1': d['macro_f1'], 'recall': [d[f'recall_{i}'] for i in range(4)]}


def kos(model, varyant):
    ad = f'{model}_abl_{varyant}'
    sonuc = Path(f'results/{ad}_eval.json')
    if sonuc.exists():
        print(ad, 'zaten tamam, atlandı')
        return
    agirlik = f'results/{ad}.pt'
    if not Path(agirlik).exists():
        calistir(f'python src/train.py --model {model} {VARYANTLAR[varyant]} --amp --name {ad}')

    ev = f'python src/evaluate.py --model {model} --weights {agirlik} --name {ad}'
    val = olcum(calistir(ev + ' --split val'))
    val_tta = olcum(calistir(ev + ' --split val --tta'))
    test = olcum(calistir(ev + ' --split test'))
    test_tta = olcum(calistir(ev + ' --split test --tta'))
    egitim = json.loads(Path(f'results/{ad}_val.json').read_text(encoding='utf-8'))
    sonuc.write_text(json.dumps({
        'model': model,
        'varyant': varyant,
        'val_macro_f1': val['macro_f1'],
        'val_macro_f1_tta': val_tta['macro_f1'],
        'test': test,
        'test_tta': test_tta,
        'epoch': egitim['epoch'],
    }, indent=2), encoding='utf-8')

    paket = [str(sonuc), f'results/{ad}_val.json', f'results/{ad}_egri.png',
             f'results/cm_{ad}.png', f'results/cm_{ad}_tta.png']
    zip_yolu = f'/content/abl_{model}_{varyant}.zip'
    with zipfile.ZipFile(zip_yolu, 'w') as z:
        for p in paket:
            if Path(p).exists():
                z.write(p)
    print(f'{ad}: val {val["macro_f1"]:.4f} (TTA {val_tta["macro_f1"]:.4f}) | test {test["macro_f1"]:.4f} (TTA {test_tta["macro_f1"]:.4f})')
    files.download(zip_yolu)

## Koşular

224 piksel koşuları T4'te yaklaşık 10–25 dakika, 384 koşusu 20–40 dakika sürer.

In [ ]:
kos(MODEL, 'temel')

In [ ]:
kos(MODEL, '384')

In [ ]:
kos(MODEL, 'guclu')

In [ ]:
kos(MODEL, 'denge')

## Bu modelin özeti

Dört modelin birleşik tablosu ve figürü, zip'ler bilgisayara alınınca `python src/ablasyon_ozet.py` ile üretilir.

In [ ]:
satirlar = []
for v in VARYANTLAR:
    yol = Path(f'results/{MODEL}_abl_{v}_eval.json')
    if yol.exists():
        d = json.loads(yol.read_text(encoding='utf-8'))
        satirlar.append({'varyant': v, 'val_f1': d['val_macro_f1'], 'val_f1_tta': d['val_macro_f1_tta'],
                         'test_f1': d['test']['macro_f1'], 'test_f1_tta': d['test_tta']['macro_f1'],
                         'recall_2': d['test']['recall'][2], 'recall_3': d['test']['recall'][3], 'epoch': d['epoch']})
pd.DataFrame(satirlar).round(4)